In [ ]:
!pip uninstall -y torch torchtext torchvision torchaudio && pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
!pip install torchtext --no-cache-dir
!pip install einops -q

In [ ]:
from google.colab import drive
drive.mount("/content/drive/")

In [ ]:
import torch
import torch.nn as nn

In [ ]:
data_set = torch.load("/content/drive/MyDrive/Group Project 2025-2026/training_data.pt")

#Test Data

In [ ]:
img_test1 = data_set['images'][0]
print(img_test1)
img_test1 = img_test1.unsqueeze(0)
print(img_test1.shape)


In [ ]:
img_test =  torch.randint(0, 2, (4,1 ,50, 200))
img_test = img_test.float()

In [ ]:
print(img_test)

In [ ]:
patch_size = 10
number_patch = (img_test.shape[2]*img_test.shape[3])//(patch_size**2)
print(number_patch)

In [ ]:
def extract_patches(image, patch_size):
  B, C, H, W = image.shape # Batch, Channel, Height, Width
  assert H % patch_size == 0
  assert W % patch_size == 0
  patchs = image.unfold (2,patch_size,patch_size).unfold(3,patch_size,patch_size)
  patchs = patchs.contiguous().view(B, C, -1, patch_size, patch_size)
  B_new, C_new, N, first_patch_size, second_patch_size = patchs.shape
  patchs = patchs.view(B_new ,N , -1)
  return patchs

In [ ]:
out = extract_patches(img_test,patch_size=patch_size)
print(out.shape)
print(out)

In [ ]:
class Patch_embedding(nn.Module):
  def  __init__(self,patch_size, num_patchs, embedding_dim):
     super().__init__()
     self.patch_size = patch_size
     self.num_patches = num_patchs
     self.embedding_dim = embedding_dim
     self.patch_embed = nn.Linear(self.patch_size**2,self.embedding_dim)
     self.cls_token = nn.Parameter(torch.randn(1, 1, self.embedding_dim))
     self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + 1, self.embedding_dim))
  def forward(self,x):
    B, N , P = x.shape
    patch_embedding = self.patch_embed(x)
    cls_tokens = self.cls_token.expand(B, -1, -1)
    tokens = torch.cat((cls_tokens, patch_embedding), dim=1)
    return tokens + self.pos_embed


In [ ]:
embed = Patch_embedding(patch_size=10,num_patchs=100,embedding_dim=128)
embedding_output = embed(out)
print(embedding_output.shape)
print(embedding_output)

In [ ]:
class MultiheadAttention(nn.Module):
  def __init__(self, embedding_dim, num_head:int=16 , dropout: float = 0) -> None:
     super().__init__()
     self.layer_norm=nn.LayerNorm(embedding_dim)
     self.mha = nn.MultiheadAttention(embed_dim=embedding_dim,
                                      num_heads=num_head,
                                      dropout=dropout,
                                      batch_first=True)
  def forward(self,x):
    x_norm = self.layer_norm(x)
    x_attention,_ = self.mha(query = x_norm,
                           key = x_norm,
                           value = x_norm,
                           need_weights = False)
    return  x + x_attention

In [ ]:
B,N,D = embedding_output.shape
attention_block = MultiheadAttention(embedding_dim = D)
attn_output = attention_block(embedding_output)
print(attn_output.shape)
print(attn_output)

In [ ]:
class MLP_Block(nn.Module):
  def __init__(self, embedding_dim,mlp_size: int = 4096 ,dropout: float = 0.1) :
     super().__init__()
     self.layer_norm=nn.LayerNorm(embedding_dim)
     self.mlp = nn.Sequential(nn.Linear(embedding_dim,mlp_size),
                              nn.GELU(),
                              nn.Dropout(dropout),
                              nn.Linear(mlp_size,embedding_dim),
                              nn.Dropout(dropout))
  def forward(self,x):
    x_norm = self.layer_norm(x)
    x_mlp = self.mlp(x_norm)
    return x + x_mlp

In [ ]:
mlpblock = MLP_Block(attn_output.shape[2])
mlp_output = mlpblock(attn_output)
print(mlp_output.shape)
print(mlp_output)

In [ ]:
class TransformerEncoderLayer(nn.Module):
  def __init__(self, embedding_dim, num_head:int=16 , attn_dropout: float = 0 , mlp_size: int =4096 , mlp_dropout: float = 0.1 ):
     super().__init__()
     self.attn = MultiheadAttention(embedding_dim,num_head,attn_dropout)
     self.mlp = MLP_Block(embedding_dim,mlp_size,mlp_dropout)
  def forward(self,x):
    x_attn = self.attn(x)
    x_mlp = self.mlp(x_attn)
    return x_mlp


In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, embedding_dim, num_layers, num_head=16, attn_dropout=0, mlp_size=4096, mlp_dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(
                embedding_dim=embedding_dim,
                num_head=num_head,
                mlp_size=mlp_size,
                attn_dropout=attn_dropout,
                mlp_dropout=mlp_dropout
            )
            for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
class ViT(nn.Module):
  def __init__(self, patch_size , num_patchs, embedding_dim,num_layer ):
     super().__init__()
     self.patch_size = patch_size
     self.embed = Patch_embedding(patch_size,num_patchs,embedding_dim)
     self.transfor = TransformerEncoder(embedding_dim ,num_layer)
  def forward(self,x):
    out1 = extract_patches(x,self.patch_size)
    out2 = self.embed(out1)
    out3 = self.transfor(out2)
    return out3


In [ ]:
vit = ViT(10,100,128,24)
dinal = vit(img_test)
print(dinal.shape)
print(dinal)

In [ ]:
layer = TransformerEncoder(128,24)
out_layer = layer(embedding_output)
print(out_layer.shape)
print(out_layer)

##############